In [1]:
import math
import pandas as pd
import numpy as numpy
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.cluster import KMeans

In [2]:
Data = pd.read_csv("bank_survey.csv")
cols = ["age", "balance", "day", "duration", "campaign", "pdays", "previous"]
data_encode = Data.drop(cols, axis=1)
data_encode = data_encode.apply(LabelEncoder().fit_transform)
data_rest = Data[cols]
Data = pd.concat([data_rest, data_encode], axis=1)

In [3]:
data_train,data_test = train_test_split(Data, test_size=0.5, random_state=4)
x_train=data_train.drop("y",axis=1)
y_train=data_train["y"]
x_test=data_test.drop("y",axis=1)
y_test=data_test["y"]

scaler=StandardScaler()
scaler.fit(x_train)
x_train=scaler.transform(x_train)
x_test=scaler.transform(x_test)

In [4]:
k_cent=8
km=KMeans(n_clusters=k_cent,max_iter=100)
km.fit(x_train)
cent=km.cluster_centers_

In [5]:
max=0
for i in range(k_cent):
  for j in range(k_cent):
    d=numpy.linalg.norm(cent[i]-cent[j])
    if d>max:
      max=d
d=max
sigma=d/math.sqrt(2*k_cent)
print(sigma)

2.0789370231156354


In [6]:
shape=x_train.shape
row=shape[0]
column=k_cent
G=numpy.empty((row,column),dtype=float)
for i in range(row):
  for j in range(column):
    dist=numpy.linalg.norm(x_train[i]-cent[j])
    G[i][j]=math.exp(-math.pow(dist,2)/math.pow(2*sigma,2))
print(G)

[[0.08088802 0.14426802 0.11835654 ... 0.0078873  0.60223569 0.15993846]
 [0.38877989 0.37214177 0.39390055 ... 0.02422623 0.23503762 0.74072122]
 [0.33658201 0.43130662 0.39284005 ... 0.02245963 0.28205702 0.53494149]
 ...
 [0.17744359 0.28993997 0.52846746 ... 0.01859411 0.17532533 0.2158771 ]
 [0.22359806 0.62492564 0.36200482 ... 0.02103559 0.21483238 0.47406731]
 [0.29591534 0.41413082 0.40205942 ... 0.02471063 0.2492273  0.65983474]]


In [7]:
GTG=numpy.dot(G.T,G)
GTG_inv=numpy.linalg.inv(GTG)
fac=numpy.dot(GTG_inv,G.T)
w=numpy.dot(fac,y_train)
print(w)

[ 1.28518759 -0.28950868 -0.14316045  0.10701872 -0.44370684  0.09996503
  0.28120195  0.03811583]


In [8]:
row=x_test.shape[0]
column=k_cent
G_test=numpy.empty((row,column),dtype=float)
for i in range(row):
  for j in range(column):
    dist=numpy.linalg.norm(x_test[i]-cent[j])
    G_test[i][j]=math.exp(-math.pow(dist,2)/math.pow(2*sigma,2))
print(G_test[0])

[0.23730573 0.38865539 0.39554015 0.56749262 0.53161731 0.02393039
 0.2269766  0.6818635 ]


In [9]:
prediction=numpy.dot(G_test,w)
prediction=0.5*(numpy.sign(prediction-0.5)+1)
score=accuracy_score(y_test,prediction)
print(score)

0.8886578784393524
